# 07: Docker para datos: PostgreSQL, Redis, Jupyter en un comando

## Receta 1: PostgreSQL listo para SQL


PostgreSQL es la base de datos que usarás en la skill de SQL. 

Aquí tienes el docker-compose.yml definitivo que usarás durante todo el track:


```yml
# docker-compose.yml - PostgreSQL para aprender SQL
services:
    postgres:
        image: postgres:16
        environment:
            POSTGRES_USER: alumno
            POSTGRES_PASSWORD: bigdatastack
            POSTGRES_DB: aprendiendo_sql
        ports: 
            - "5432:5432"
        volumes: 
            - postgres_data:/var/lib/postgresql/data
            - ./init.sql:/docker-entrypoint-initdb.d/init.sql # Script inicial

volumes:
    postgres_data: 
```

Línea por línea:

- `image: postgres:16` → Imagen oficial de PostgreSQL 16.
- `POSTGRES_USER/PASSWORD/DB` → Crea user "alumno", password "bigdatastack", DB "aprendiendo_sql" al primer arranque.
- `ports: - "5432:5432"` → PostgreSQL accesible en localhost:5432 desde tu máquina.
- `postgres_data:/var/lib/postgresql/data` → Named volume: los datos sobreviven a docker compose down.
- `./init.sql:/docker-entrypoint-initdb.d/init.sql` → TRUCO: cualquier .sql en esa carpeta se ejecuta automáticamente al crear la DB por primera vez. Perfecto para seed data.

Este docker-compose.yml es el que usarás durante toda la skill de SQL.



El truco del /docker-entrypoint-initdb.d/ es brillante:

1. cualquier archivo .sql que pongas en esa carpeta se ejecuta AUTOMÁTICAMENTE la primera vez que PostgreSQL arranca. 
2. Perfecto para crear tablas y cargar datos de ejemplo sin esfuerzo manual.



**CUIDADO: **
- PostgreSQL SOLO ejecuta los scripts de initdb.d cuando el volumen de datos está vacío (primera vez). 
- Si ya tenías el contenedor corriendo de lecciones anteriores, el pgdata ya existe y el init.sql se ignora silenciosamente. 
- **Solución:** ejecuta "`docker compose down -v`" para borrar el volumen y empezar limpio. La -v es clave — sin ella solo para el contenedor pero mantiene los datos.



```sql
-- Archivo: init.sql (se ejecuta automáticamente al crear la DB)
CREATE TABLE IF NOT EXISTS clientes (
    id SERIAL PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL,
    email VARCHAR(250) UNIQUE NOT NULL,
    nombre VARCHAR(50),
    fecha_registro DATE DEFAULT CURRENT_DATE,
);

INSERT INTO clientes (nombre, email, ciudad) VALUES
('Ana García', 'ana@ejemplo.com', 'Madrid'),
('Carlos López', 'carlos@ejemplo.com', 'Barcelona'),
('María Fernández', 'maria@ejemplo.com', 'Valencia');
```


- `CREATE TABLE IF NOT EXISTS clientes (...)` → Crea la tabla solo si no existe. Seguro de ejecutar múltiples veces.
- `id SERIAL PRIMARY KEY` → ID autoincremental, clave primaria.
- `email VARCHAR(200) UNIQUE NOT NULL` → Email obligatorio y único (no permite duplicados).
- `fecha_registro DATE DEFAULT CURRENT_DATE` → Si no se especifica, usa la fecha de hoy.
- `INSERT INTO clientes...VALUES...` → Inserta 3 filas de ejemplo para tener datos desde el minuto 0.

Este archivo se ejecuta UNA sola vez: cuando PostgreSQL inicializa la DB por primera vez. Si el volumen ya tiene datos, se ignora.

## Receta 2: Redis para cache de datos


Redis es un almacén de datos en memoria ultrarrápido. 
- En data engineering se usa para cachear resultados de queries lentas, almacenar estados temporales de pipelines, y como broker de mensajes ligero. 
- Levantarlo con Docker es trivial:



```yml
# Añadir Redis al docker-compose.yml
services:
    redis:
        image: redis:7-alpine
        ports:
            - "6379:6379"
        volumes:
            - redis_data:/data
        command: redis-server --appendonly yes # Persistir datos a disco
```

- `image: redis:7-alpine` → Redis 7 sobre Alpine Linux (~7 MB). Alpine = imagen mínima, ideal para servicios ligeros.
- `ports: - "6379:6379"` → Puerto estándar de Redis mapeado a tu máquina.
- `redis_data:/data` → Named volume para persistir los datos de Redis.
- `command: redis-server --appendonly yes` → Sobreescribe el CMD por defecto. `--appendonly yes` activa el log de escrituras (AOF) para que los datos sobrevivan a reinicios.

Sin `--appendonly yes`, Redis solo guarda en memoria y se pierde al parar el contenedor.



## Receta 3: Jupyter Notebook con ciencia de datos


```yml
# Jupyter con pandas, numpy, matplotlib preinstalados
services:
    jupyter:
        image: jupyter/scipy-notebook:latest
        ports:
            - "8888:8888" 
        volumes: 
            - ./notebooks:/home/jovyan/work # Tus notebooks persistents
        environment: 
            JUPYTER_TOKEN: "bigdatastack" # Token para acceder
```

- `image: jupyter/scipy-notebook:latest` → Imagen oficial de Jupyter con scipy, pandas, numpy, matplotlib preinstalados.
- `ports: - "8888:8888"` → Jupyter sirve en el puerto 8888. Accedes desde tu navegador.
- `./notebooks:/home/jovyan/work` → Bind mount: los notebooks que crees se guardan en tu carpeta local ./notebooks. "jovyan" es el usuario por defecto de las imágenes Jupyter.
- `JUPYTER_TOKEN: "bigdatastack"` → Token de acceso. Al abrir http://localhost:8888 te pedirá este token.

Alternativa: usa `jupyter/datascience-notebook` para tener también R y Julia.

**Consejo de senior:**

1. Las imágenes jupyter/scipy-notebook y jupyter/datascience-notebook incluyen pandas, numpy, scipy, matplotlib, seaborn y más — todo preconfigurado y listo para usar. 
2. No necesitas instalar NADA localmente. 
3. ¿Quieres probar una librería nueva? La añades al Dockerfile. }
4. ¿Rompiste algo? docker compose down y empiezas de cero.



## Todo junto: el stack de datos completo

Ahora combinemos PostgreSQL + Redis + Jupyter en un solo docker-compose.yml.

Este es el tipo de setup que usarás en tu día a día como data engineer:


```yml
services:
    postgres:
        image: postgres:16
        environment:
            POSTGRES_USER: datos
            POSTGRES_PASSWORD: secreto123
            POSTGRES_DB: analytics
        ports:
            - "5432:5432"
        volumes:
            - pgdata:/var/lib/postgresql/data

    redis:
        image: redis:7-alpine
        ports: 
            - "6379:6379"
        volumes:
            - redis_data:/data

    jupyter:
        image: jupyter/scipy-notebook:latest
        ports:
            - "8888:8888"
        volumes:
            - ./notebooks:/home/jovyan/work
        environment:
            JUPYTER_TOKEN: "datos2024"
        depends_on:
            - postgres
            - redis

volumes:
    pgdata:
    redis_data: 
```

## Conectar desde Python a tus servicios Docker


Una vez que tus servicios están corriendo, puedes conectarte a ellos desde Python en tu ordenador local como si fueran servicios normales. 
1. Recuerda: los puertos que mapeaste (5432, 6379, 8888) están accesibles en localhost.



```python
# Conectar a PostgreSQL desde Python (necesitas: pip install psycopg2-binary)
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    user="datos",
    password="secreto123",
    database="analytics"
)

cursor = con.cursor()
cursor.execute("SELECT version();")
print(f"PostgreSQL: {cursor.fetchone()[0]}")
conn.close()
```


- `import psycopg2` → Driver de PostgreSQL para Python. Necesita `pip install psycopg2-binary`.
- `host="localhost"` → Desde TU máquina, la DB está en localhost (el puerto está mapeado).
- `port=5432` → Puerto que mapeaste en docker-compose.
- `conn = psycopg2.connect(...)` → Abre una conexión TCP a PostgreSQL.
- `cursor.execute("SELECT version();")` → Ejecuta una query para verificar la conexión.
- `conn.close()` → SIEMPRE cierra la conexión cuando termines.

Recuerda: `host="localhost"` solo desde tu PC. Dentro de otro contenedor sería `host="postgres"`.

```python
# Conectar a Redis desde Python (necesitas: pip install redis)
import redis

r = redis.Redis(
    host="localhost",
    port=6379,
    decode_responses=True
)

r.set("pipeline status", "ejecutando")
print(f"Estado del pipeline: {r.get("pipeline_status")}")
```

- `import redis` → Cliente Redis para Python. Necesita `pip install redis`.
- `redis.Redis(host='localhost', port=6379, decode_responses=True)` → Conecta a Redis. `decode_responses=True` convierte bytes a strings automáticamente.
- `r.set('pipeline_status', 'ejecutando')` → Guarda un par clave-valor en Redis.
- `r.get('pipeline_status')` → Recupera el valor. Redis es ultrarrápido (<1ms).

En pipelines reales, Redis se usa para cachear resultados de queries lentas y almacenar el estado de ejecución de ETLs.



Recuerda: desde TU ordenador, usas localhost:5432 para conectarte a PostgreSQL. 

1. Pero desde OTRO contenedor dentro del mismo docker-compose, usarías postgres:5432 (el nombre del servicio). 
2. Este es el error más común de principiantes — confundir cuándo usar localhost y cuándo el nombre del servicio.



## pgAdmin: gestión visual de PostgreSQL


pgAdmin es una interfaz web para gestionar PostgreSQL visualmente. 

También lo puedes levantar con Docker — un servicio más en tu compose:



```yml
# Añadir pgAdmin al docker-compose.yml
services:
    pgadmin:
        images: dpage/pgadmin4:latest
        environment:
            PGADMIN_DEFAULT_EMAIL: admin@bigdatastack.com
            PGADMIN_DEFAULT_PASSWORD: admin123
        ports:
            - "8080:80"
        depends_on:
            - postgres

# pgAdmin en http://localhost:8080 — gestión visual de tu DB

```

- `image: dpage/pgadmin4:latest` → Imagen oficial de pgAdmin 4 (interfaz web para PostgreSQL).
- `PGADMIN_DEFAULT_EMAIL` → Email para el login de pgAdmin.
- `PGADMIN_DEFAULT_PASSWORD` → Password para el login.
- `ports: - "8080:80"` → pgAdmin sirve en el puerto 80 internamente, lo mapeamos al 8080 de tu máquina.
- `depends_on: - postgres` → pgAdmin arranca después de PostgreSQL.

Al conectar pgAdmin a la DB, usa hostname "postgres" (nombre del servicio), NO "localhost". Ambos están en la misma red Docker.



Consejo de senior: al conectar pgAdmin a PostgreSQL dentro de Docker, el hostname NO es localhost — es "postgres" (el nombre del servicio en docker-compose). El puerto es 5432. Este detalle te ahorrará 30 minutos de frustración la primera vez.
